# Tutorial 4 -- Containers

Wind-tunnel post-processing runs the same pipeline many times -- once per wind direction, once per recurrence period, once per case label. A **`Container[K, V]`** is the v3 way to hold those results:

- keyed by a hashable Pydantic case-parameters object;
- supports `filter_by`, `join_by(callback)` for partitioning;
- supports `map_values(pipeline, pool=...)` to run a pipeline over every entry, sequentially or in parallel.

This is the same shape as the legacy `HFPIAnalysisResults`; the new `Container` is its generic version.

In [1]:
from pydantic import BaseModel, ConfigDict
import numpy as np

from cfdmod import (
    Container,
    SurfaceDataSource, TimeAxis, Topology, ElementMeta,
    MemoryFieldStore,
)

class WindCase(BaseModel):
    """Hashable key for a per-direction simulation."""
    model_config = ConfigDict(frozen=True)
    direction: int  # degrees
    recurrence_period: float  # years

## 1. Populate a container with synthetic per-direction Cp

Eight directions x two recurrence periods = 16 cases.

In [2]:
rng = np.random.default_rng(0)
verts = np.array([[0, 0, 0], [1, 0, 0], [0, 1, 0]], dtype=np.float64)
tris = np.array([[0, 1, 2]], dtype=np.int32)

items = {}
for direction in range(0, 360, 45):
    for rp in (50.0, 100.0):
        cp = rng.normal(loc=-0.5 + 0.01 * direction, scale=0.1, size=(1, 200))
        ds = SurfaceDataSource(
            time=TimeAxis(initial_time=0.0, timestep_size=0.001, n_timesteps=200),
            topology=Topology.triangles(tris, verts),
            elements=ElementMeta(area=np.array([0.5])),
            fields=MemoryFieldStore({"cp": cp}),
        )
        items[WindCase(direction=direction, recurrence_period=rp)] = ds

container: Container[WindCase, SurfaceDataSource] = Container(items=items)
print(f"container has {len(container)} entries")

container has 16 entries


## 2. Partition by a key derivation

`join_by(callback)` groups entries that share a derived key into their own sub-containers. Canonical uses:

- one sub-container per direction (callback: `lambda k: k.direction`);
- one sub-container per recurrence period.

Each sub-container is itself a `Container[K, V]` -- partitions compose.

In [3]:
by_direction = container.join_by(lambda k: k.direction)
print("directions:    ", sorted(by_direction))
print("d=0 entries:   ", len(by_direction[0]))

by_period = container.join_by(lambda k: k.recurrence_period)
print("periods:       ", sorted(by_period))
print("50yr entries:  ", len(by_period[50.0]))

directions:     [0, 45, 90, 135, 180, 225, 270, 315]
d=0 entries:    2
periods:        [50.0, 100.0]
50yr entries:   8


## 3. Filter by a predicate

In [4]:
windward = container.filter_by(lambda k: 0 <= k.direction <= 90 and k.recurrence_period == 100.0)
print("windward (0-90 deg, 100yr):")
for key in windward:
    print("  ", key)

windward (0-90 deg, 100yr):
   direction=0 recurrence_period=100.0
   direction=45 recurrence_period=100.0
   direction=90 recurrence_period=100.0


## 4. Run a pipeline over every entry

`map_values(callable, pool=None)` applies the function to every value. Pass a `Pool` for parallel fanout; leave it `None` for sequential (the default).

In [5]:
from cfdmod.ops.data_source_create import compute_statistics, StatisticsParams
from functools import partial

stats_pipe = partial(compute_statistics, p=StatisticsParams(field="cp", kinds=["mean", "rms", "peak_max"]))
stats_container = container.map_values(stats_pipe)

print("per-direction max:")
for key in sorted(stats_container, key=lambda k: (k.direction, k.recurrence_period)):
    if key.recurrence_period == 100.0:
        peak = stats_container[key].fields.read("peak_max")[0]
        print(f"  dir={key.direction:3d}  peak_max(cp) = {peak:+.3f}")

per-direction max:
  dir=  0  peak_max(cp) = -0.193
  dir= 45  peak_max(cp) = +0.197
  dir= 90  peak_max(cp) = +0.687
  dir=135  peak_max(cp) = +1.087
  dir=180  peak_max(cp) = +1.549
  dir=225  peak_max(cp) = +2.016
  dir=270  peak_max(cp) = +2.453
  dir=315  peak_max(cp) = +2.906


## 5. Reduce a partition

Worst-case Cp per direction across recurrence periods:

In [6]:
worst_per_direction = {}
for direction, sub in by_direction.items():
    # compute statistics on each entry, then take the highest peak_max.
    sub_stats = sub.map_values(stats_pipe)
    worst = max(
        (sub_stats[k].fields.read("peak_max")[0] for k in sub_stats),
    )
    worst_per_direction[direction] = worst

for d in sorted(worst_per_direction):
    print(f"  dir={d:3d}  worst peak_max = {worst_per_direction[d]:+.3f}")

  dir=  0  worst peak_max = -0.193
  dir= 45  worst peak_max = +0.197
  dir= 90  worst peak_max = +0.687
  dir=135  worst peak_max = +1.124
  dir=180  worst peak_max = +1.569
  dir=225  worst peak_max = +2.016
  dir=270  worst peak_max = +2.453
  dir=315  worst peak_max = +2.947


## 6. Parallel fanout (optional)

Pass a `multiprocessing.Pool` to `map_values` and the pipeline runs concurrently. The same `pool=None` no-op default keeps tests and notebooks fast.

```python
from multiprocessing import Pool
with Pool(4) as pool:
    stats = container.map_values(stats_pipe, pool=pool)
```

(Skipped here to keep the notebook deterministic.)

## Recap

- `Container[K, V]`: hashable-keyed, frozen, functional updates via `with_item` / `merge`.
- `join_by`: partition by a derived key.
- `filter_by`: predicate-based sub-container.
- `map_values(pipe, pool=None)`: run any pipeline over every entry.

End of the four-tutorial intro. Domain notebooks (`envelope`, `global_forces`, `psd_profile_velocity`, `hfpi_*`) demonstrate the recipes in their full production setting.